## Read Data from Bronze Layer

In [0]:
%run ./test

## Load Bronze Data from Unity Catlog

In [0]:
def load_data(table):
  return spark.read.table(table)


## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_load_data'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Get Star Players from Common Player Info

By filtering the greatest 75th anniversay team players

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col

def get_star_players(common_player_info_df: DataFrame, inactive_players_df: DataFrame):
    all_star_players_df = common_player_info_df\
        .where(col("greatest_75_flag") == "Y")\
            .withColumn("player_id", col("person_id"))\
                .drop("person_id")\
                    .select("player_id", "first_name", "last_name", "display_first_last", "position", "from_year", "to_year")

    invalid_star_players = all_star_players_df.join(inactive_players_df, on="player_id").groupBy("player_id").count().where("count <= 10").select("player_id")

    return all_star_players_df.join(invalid_star_players, on="player_id", how="left_anti")

## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_get_star_players'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Process Games with Star Absence

Get the games that the star players are off

In [0]:
def process_game_with_star_absence(star_players_df: DataFrame, inactive_players_df: DataFrame):
    return star_players_df.join(inactive_players_df, on="player_id")


## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_process_game_with_star_absence'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Process Games Statistics with Star Players Absence

Only consider the regular season games

In [0]:
def process_star_absence_games(game_with_star_absence_df: DataFrame, game_df: DataFrame):
    return game_with_star_absence_df.join(game_df, on="game_id").where(col("season_type") == "Regular Season")

## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_process_star_absence_games'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Process Games with Star Absence

In [0]:
def process_game(common_player_info_df: DataFrame, game_df: DataFrame, inactive_players_df: DataFrame):
    star_df = common_player_info_df.transform(get_star_players, inactive_players_df)
    game_with_star_absence_df = star_df.transform(process_game_with_star_absence, inactive_players_df)
    game_stats_with_star_absence_df = game_with_star_absence_df.transform(process_star_absence_games, game_df)\
        .withColumn("star_team_id", col("team_id"))\
            .drop("team_id")\
                .select("game_id", "star_team_id", "team_id_home", "wl_home", "team_id_away", "wl_away", "player_id")

    return game_stats_with_star_absence_df

INACTIVE_PLAYERS_TABLE = "nba_insights.bronze.inactive_players"
COMMON_PLAYER_INFO_TABLE = "nba_insights.bronze.common_player_info"
GAME_TABLE = "nba_insights.bronze.games"

inactive_players_df = load_data(INACTIVE_PLAYERS_TABLE)
common_player_info_df = load_data(COMMON_PLAYER_INFO_TABLE)
game_df = load_data(GAME_TABLE)

star_players_df = common_player_info_df.transform(get_star_players, inactive_players_df)
game_stats_with_star_absence_df = process_game(common_player_info_df, game_df, inactive_players_df)
display(game_stats_with_star_absence_df)

## Star Franchise Games History
All games that the stars played in the history

In [0]:
from pyspark.sql.functions import col, substring

def process_star_franchise_games_history(game_df: DataFrame, inactive_players_df: DataFrame, star_players_df: DataFrame):
    games_with_years = game_df.withColumn(
        "game_year", substring(col("game_date"), 1, 4).cast("int")
    )

    # It's possible that the star players may not be off during an entire season
    player_team_history_df = inactive_players_df.select("player_id", "team_id").distinct()

    star_teams_df = player_team_history_df.join(
        star_players_df.select("player_id", "from_year", "to_year"), 
        on="player_id"
    )

    team_games_home = star_teams_df.join(
        games_with_years,
        on = (star_teams_df.team_id == games_with_years.team_id_home) &
        (games_with_years.game_year >= star_teams_df.from_year) &
        (games_with_years.game_year <= star_teams_df.to_year)
    ).select(
        col("player_id"),
        col("game_id"),
        col("team_id_home").alias("team_id"),
        col("wl_home").alias("team_outcome")
    )

    team_games_away = star_teams_df.join(
        games_with_years,
        (star_teams_df.team_id == games_with_years.team_id_away) &
        (games_with_years.game_year >= star_teams_df.from_year) &
        (games_with_years.game_year <= star_teams_df.to_year)
    ).select(
        col("player_id"),
        col("game_id"),
        col("team_id_away").alias("team_id"),
        col("wl_away").alias("team_outcome")
    )

    return team_games_home.union(team_games_away).dropDuplicates(["player_id", "game_id"])

star_franchise_games_history_df = process_star_franchise_games_history(game_df, inactive_players_df, star_players_df)

## Persist Silver Layer

In [0]:
star_players_df.write.mode("overwrite").saveAsTable("nba_insights.silver.star_players")

game_stats_with_star_absence_df.write.mode("overwrite").saveAsTable("nba_insights.silver.games_stats_with_stars_absence")

star_franchise_games_history_df.write.mode("overwrite").saveAsTable("nba_insights.silver.star_franchise_games_history")